# Módulo 1 · Práctica: tu primer modelo de lenguaje

En este notebook construyes, desde cero y con NumPy, un **modelo de bigramas de caracteres** que inventa nombres en español. Es el mismo esquema que siguen Claude, Gemini o Grok: predecir el siguiente token, medir la pérdida con entropía cruzada y generar muestreando. Solo cambia cómo se calcula la probabilidad.

**Cómo trabajar:** ejecuta las celdas en orden (Mayús + Intro). Las celdas marcadas con `# TU CÓDIGO` son ejercicios: complétalas y ejecuta la celda de comprobación que va detrás.

| Parte | Qué haces |
| --- | --- |
| 0 | Repaso exprés de Python y NumPy |
| 1 | Datos y vocabulario |
| 2 | Contar bigramas y normalizar |
| 3 | Generar nombres |
| 4 | Pérdida, perplejidad y validación |
| 5 | Temperatura y suavizado |
| 6 | Retos |

## Parte 0 · Repaso exprés de NumPy

Solo necesitas cuatro ideas: crear arrays, indexar, sumar por ejes y el *broadcasting*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[1., 2., 3.],
              [4., 5., 6.]])
print(A.shape)                       # (2, 3): 2 filas, 3 columnas
print(A[1, 2])                       # fila 1, columna 2 -> 6.0
print(A.sum(axis=1))                 # suma de cada fila -> [ 6. 15.]
print(A.sum(axis=1, keepdims=True))  # misma suma, pero con forma (2, 1)
print(A / A.sum(axis=1, keepdims=True))  # broadcasting: cada fila dividida por su total

**Ejercicio 0.** Crea una matriz `B` de 3×3 con los números 1 a 9 (pista: `np.arange` y `.reshape`) y normaliza sus **columnas** para que cada una sume 1.

In [ ]:
# TU CÓDIGO
B = None
B_norm = None

In [ ]:
# Comprobación
assert B is not None and B.shape == (3, 3), "B debe ser 3x3"
assert np.allclose(B_norm.sum(axis=0), 1), "Cada columna debe sumar 1"
print("Correcto")

## Parte 1 · Datos y vocabulario

Usamos una lista de nombres españoles, sin tildes salvo la ñ. Añadimos el token especial `.` para marcar inicio y fin.

In [ ]:
nombres = [
    'alejandro', 'antonio', 'manuel', 'jose', 'francisco', 'david', 'juan', 'javier', 'daniel', 'carlos',
    'jesus', 'miguel', 'rafael', 'pablo', 'pedro', 'angel', 'sergio', 'fernando', 'jorge', 'luis',
    'alberto', 'alvaro', 'adrian', 'diego', 'raul', 'enrique', 'ramon', 'ivan', 'ruben', 'oscar',
    'andres', 'vicente', 'joaquin', 'santiago', 'victor', 'eduardo', 'mario', 'roberto', 'jaime', 'ignacio',
    'marcos', 'alfonso', 'jordi', 'hugo', 'ricardo', 'salvador', 'guillermo', 'gabriel', 'emilio', 'gonzalo',
    'julio', 'julian', 'martin', 'tomas', 'marc', 'agustin', 'nicolas', 'samuel', 'felix', 'ismael',
    'cristian', 'aitor', 'joan', 'josep', 'lucas', 'mateo', 'leo', 'bruno', 'enzo', 'izan',
    'alex', 'gael', 'thiago', 'marco', 'dario', 'unai', 'iker', 'eric', 'rodrigo', 'aaron',
    'nil', 'pau', 'oliver', 'liam', 'adam', 'biel', 'mauro', 'saul', 'cesar', 'lorenzo',
    'benito', 'esteban', 'felipe', 'gregorio', 'hector', 'horacio', 'jacinto', 'leandro', 'lisandro', 'matias',
    'nestor', 'octavio', 'patricio', 'ramiro', 'sebastian', 'teodoro', 'ulises', 'valentin', 'xavier', 'yago',
    'zacarias', 'aurelio', 'baltasar', 'ceferino', 'domingo', 'elias', 'fabian', 'gaspar', 'hilario', 'iñigo',
    'nuño', 'toño', 'maria', 'carmen', 'ana', 'isabel', 'laura', 'cristina', 'marta', 'lucia',
    'paula', 'elena', 'sara', 'raquel', 'rosa', 'pilar', 'manuela', 'concepcion', 'mercedes', 'julia',
    'beatriz', 'nuria', 'silvia', 'irene', 'patricia', 'rocio', 'andrea', 'montserrat', 'monica', 'alba',
    'sonia', 'sandra', 'marina', 'angela', 'sofia', 'alicia', 'susana', 'yolanda', 'claudia', 'natalia',
    'margarita', 'eva', 'inmaculada', 'noelia', 'veronica', 'esther', 'carla', 'nerea', 'daniela', 'alejandra',
    'martina', 'valeria', 'chloe', 'emma', 'olivia', 'jimena', 'vega', 'lola', 'carlota', 'mia',
    'valentina', 'aitana', 'noa', 'alma', 'triana', 'candela', 'ainhoa', 'ines', 'adriana', 'aroa',
    'blanca', 'celia', 'diana', 'elsa', 'fatima', 'gloria', 'helena', 'iria', 'lara', 'lidia',
    'lorena', 'luna', 'maite', 'mar', 'miriam', 'nadia', 'nieves', 'noemi', 'olga', 'paloma',
    'regina', 'rebeca', 'salma', 'tamara', 'teresa', 'ursula', 'victoria', 'ximena', 'zoe', 'aurora',
    'begoña', 'catalina', 'dolores', 'encarnacion', 'francisca', 'gema', 'juana', 'leire', 'macarena', 'nayara',
    'oihane', 'piedad', 'remedios', 'soledad', 'trinidad', 'vanesa', 'yaiza', 'amparo', 'araceli', 'asuncion',
    'cecilia', 'consuelo', 'emilia', 'esperanza', 'estefania', 'eugenia', 'guadalupe', 'josefa', 'lourdes', 'maribel',
    'purificacion', 'rosario', 'virginia', 'almudena', 'ariadna', 'berta', 'carolina', 'debora', 'estela', 'fernanda',
    'graciela', 'jazmin', 'magdalena', 'ofelia', 'pepa', 'rut', 'sabina', 'tatiana', 'yasmina', 'zaira',
    'ainara', 'arantxa', 'iratxe', 'naia', 'uxue', 'ane', 'june', 'maialen', 'itziar', 'alonso',
    'ernesto', 'marcelo', 'bernardo', 'cayetano', 'rosalia', 'anabel', 'candido', 'clemente', 'efren', 'genaro',
    'isidro', 'leonardo', 'maximo', 'norberto', 'rogelio', 'silvestre', 'tadeo', 'amelia', 'bianca', 'cintia',
    'dafne', 'erika', 'flor', 'ivonne', 'judith', 'karen', 'leticia', 'micaela', 'nora', 'perla',
    'rita', 'selena', 'telma', 'vera', 'wendy', 'abril',
]
print(len(nombres), 'nombres')

In [ ]:
vocab = list(".abcdefghijklmnñopqrstuvwxyz")
V = len(vocab)
stoi = {c: i for i, c in enumerate(vocab)}   # carácter -> entero
itos = {i: c for i, c in enumerate(vocab)}   # entero -> carácter
print(V, "tokens")

def tokenizar(palabra):
    return [stoi[c] for c in "." + palabra + "."]

print(tokenizar("luna"))

In [ ]:
# Separamos un 15 % para validación (con semilla fija para que sea reproducible)
rng = np.random.default_rng(7)
idx = rng.permutation(len(nombres))
n_val = round(0.15 * len(nombres))
val = [nombres[i] for i in idx[:n_val]]
train = [nombres[i] for i in idx[n_val:]]
print(len(train), "entrenamiento ·", len(val), "validación")

## Parte 2 · Contar y normalizar

`N[i, j]` cuenta cuántas veces el token `j` sigue al token `i`. Después `P[i, j] = N[i, j] / Σₖ N[i, k]`.

**Ejercicio 1.** Rellena la matriz de conteos `N` recorriendo los bigramas de cada nombre de `train`.

In [ ]:
N = np.zeros((V, V))
for palabra in train:
    t = tokenizar(palabra)
    # TU CÓDIGO: recorre los pares consecutivos (t[k], t[k+1]) y suma 1 en N
    pass

In [ ]:
# Comprobación
assert N.sum() == sum(len(p) + 1 for p in train), "Cada nombre de longitud L aporta L + 1 bigramas"
print("Correcto. Bigramas contados:", int(N.sum()))
print("Veces que un nombre empieza por 'a':", int(N[stoi['.'], stoi['a']]))

**Ejercicio 2.** Calcula la matriz de probabilidades `P` normalizando cada fila. Una línea basta.

In [ ]:
# TU CÓDIGO
P = None

In [ ]:
# Comprobación
assert P is not None and P.shape == (V, V)
assert np.allclose(P[N.sum(axis=1) > 0].sum(axis=1), 1), "Cada fila con datos debe sumar 1"
print("Correcto")

In [ ]:
# Visualización de la matriz de transición
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.sqrt(P), cmap="Blues")
ax.set_xticks(range(V)); ax.set_xticklabels(vocab)
ax.set_yticks(range(V)); ax.set_yticklabels(vocab)
ax.set_xlabel("siguiente"); ax.set_ylabel("anterior")
ax.set_title("P(siguiente | anterior), escala raíz cuadrada")
plt.show()

**Pregunta.** Mira la fila `.` (cómo empiezan los nombres) y la columna `.` (con qué letra terminan). ¿Qué patrones del español ves?

## Parte 3 · Generar nombres

Repetimos: mirar la fila del token actual, sortear el siguiente con `rng.choice(V, p=fila)`, parar al salir `.`.

**Ejercicio 3.** Completa la función `generar`.

In [ ]:
def generar(P, rng, max_len=20):
    nombre = ""
    i = stoi["."]
    for _ in range(max_len):
        # TU CÓDIGO: sortea j con las probabilidades de la fila i
        # si j es el token '.', termina; si no, añade itos[j] y actualiza i
        break
    return nombre

gen_rng = np.random.default_rng(42)
print([generar(P, gen_rng) for _ in range(15)])

In [ ]:
# Comprobación
test = [generar(P, np.random.default_rng(s)) for s in range(200)]
assert all(set(n) <= set(vocab[1:]) for n in test), "Los nombres no deben contener '.'"
assert sum(len(n) > 0 for n in test) > 150, "La mayoría de nombres deberían tener letras"
nuevos = sum(n not in nombres for n in test)
print(f"Correcto. {nuevos} de 200 nombres generados no estaban en el corpus.")

## Parte 4 · Pérdida y perplejidad

La pérdida es la log-verosimilitud negativa media:

$$\mathcal{L} = -\frac{1}{N}\sum_t \log P(x_t \mid x_{t-1}), \qquad \mathrm{PPL} = e^{\mathcal{L}}$$

**Ejercicio 4.** Implementa `perdida(P, palabras)`.

In [ ]:
def perdida(P, palabras):
    # TU CÓDIGO: suma -log P[i, j] de todos los bigramas y divide por cuántos hay
    return None

print("Entrenamiento:", perdida(P, train))
print("Validación:   ", perdida(P, val))

In [ ]:
# Comprobación con el ejemplo del documento: corpus ana, ala, luna -> 0,619 nats
mini = ["ana", "ala", "luna"]
Nm = np.zeros((V, V))
for p in mini:
    t = tokenizar(p)
    for a, b in zip(t, t[1:]):
        Nm[a, b] += 1
Pm = Nm / np.maximum(Nm.sum(axis=1, keepdims=True), 1)
assert abs(perdida(Pm, mini) - 0.6190) < 1e-3, "Revisa la fórmula"
print("Correcto. Perplejidad del ejemplo:", round(np.exp(perdida(Pm, mini)), 3))

Si la pérdida de validación sale `inf`, no es un error tuyo: algún bigrama de validación nunca apareció en entrenamiento. Lo arreglamos en la parte 5.

## Parte 5 · Temperatura y suavizado

In [ ]:
def suavizar(N, alpha):
    return (N + alpha) / (N + alpha).sum(axis=1, keepdims=True)

for alpha in [0.01, 0.1, 0.5, 1, 2, 5]:
    Pa = suavizar(N, alpha)
    print(f"alpha = {alpha:<5} entrenamiento {perdida(Pa, train):.3f}  validación {perdida(Pa, val):.3f}  PPL val {np.exp(perdida(Pa, val)):.2f}")
print(f"Modelo uniforme: PPL = {V}")

**Pregunta.** ¿Por qué la pérdida de entrenamiento sube siempre con α, mientras que la de validación primero baja y luego sube? Relaciónalo con el sobreajuste.

In [ ]:
def con_temperatura(P, T):
    Q = P ** (1 / T)            # equivale a softmax(log P / T)
    return Q / Q.sum(axis=1, keepdims=True)

Ps = suavizar(N, 0.5)
for T in [0.3, 1.0, 2.0]:
    r = np.random.default_rng(0)
    print(f"T = {T}:", [generar(con_temperatura(Ps, T), r) for _ in range(8)])

**Ejercicio 5.** Demuestra numéricamente que `P ** (1/T)` normalizado coincide con `softmax(log P / T)`. Escribe una función `softmax(z, T)` estable (restando el máximo) y compara ambas en la fila de la `a`.

In [ ]:
def softmax(z, T=1.0):
    # TU CÓDIGO
    return None

fila = Ps[stoi["a"]]
print(np.allclose(softmax(np.log(fila), 0.7), con_temperatura(Ps, 0.7)[stoi["a"]]))

## Parte 6 · Retos

1. **Trigramas.** Construye un modelo con contexto de dos caracteres (tabla de V × V × V). Compara su perplejidad de validación con la de bigramas. ¿Cuántas casillas tiene la tabla y cuántas quedan a cero?
2. **Tu corpus.** Cambia `nombres` por palabras de un libro en español (o tus mensajes). ¿Qué perplejidad obtienes?
3. **Probabilidad de tu nombre.** ¿Qué probabilidad le asigna el modelo? ¿Qué transición es la más improbable?
4. **Pensar en grande.** Un LLM con vocabulario de 100.000 tokens tendría una tabla de bigramas de 10¹⁰ casillas. Explica por qué una red neuronal con muchos menos parámetros por contexto puede hacerlo mucho mejor (adelanto de los módulos 2 y 9).

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 0
# B = np.arange(1, 10, dtype=float).reshape(3, 3);  B_norm = B / B.sum(axis=0, keepdims=True)
# Ejercicio 1 (dentro del bucle)
#     for a, b in zip(t, t[1:]):
#         N[a, b] += 1
# Ejercicio 2
# P = N / N.sum(axis=1, keepdims=True)
# Ejercicio 3 (dentro del bucle)
#     j = rng.choice(V, p=P[i])
#     if j == stoi["."]: break
#     nombre += itos[j];  i = j
# Ejercicio 4
#     ll = [np.log(P[a, b]) for p in palabras for a, b in zip(tokenizar(p), tokenizar(p)[1:])]
#     return -np.mean(ll)
# Ejercicio 5
#     z = np.asarray(z) / T;  e = np.exp(z - z.max());  return e / e.sum()